# Full M9 Saleability Mechanism

- Full M9 pipeline
- Saleability score generation
- Final Excel export for upcoming data (`product_arrival.xlsx`)

The final scoring engine is named **`SSCORER`**


In [1]:
# SHARED SETUP — run this cell first
#EN ÖNEMLİ YER:
# !!!!Reproduces the EXACT same train/test split and trained pipelines as 01_ols_regression.ipynb  (random_state=42, stratify=URUN ALT GRUBU)
import numpy as np
import pandas as pd
import scipy.sparse as sp
import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
import statsmodels.api as sm
from scipy.stats import f as f_dist

# Data
df = pd.read_excel("../../data/01_raw/tomodel.xlsx").copy()

TARGET           = "avg_qty_when_active"
CAT_COLS         = ["URUN ALT GRUBU", "LifeStyleGroup", "ColorGroup"]
ORD_COLS         = ["size_availability_award", "discount_group"]
NUM_COLS         = ["ETIKET"]
df["ETIKET"]     = np.log1p(df["ETIKET"])
ALPHA            = 0.05
TREAT_ORD_AS_NUM = False

X_full = df.copy()
y_full = df[TARGET].astype(float)

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=df["URUN ALT GRUBU"])

y_train = np.log1p(y_train_raw)
y_test  = np.log1p(y_test_raw)

strat_labels = X_train["URUN ALT GRUBU"]
min_cls      = strat_labels.value_counts().min()
N_SPLITS     = 2 if min_cls < 5 else 5
N_REPEATS    = 5

# Helpers
def mape_safe(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask   = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])))

def evaluate(y_true_log, y_pred_log):
    y_true_orig = np.expm1(np.asarray(y_true_log, dtype=float))
    y_pred_orig = np.expm1(np.asarray(y_pred_log, dtype=float))
    return {
        "MAE":     float(mean_absolute_error(y_true_orig, y_pred_orig)),
        "RMSE":    float(np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))),
        "MAPE":    mape_safe(y_true_orig, y_pred_orig),
        "R2":      float(r2_score(y_true_log,  y_pred_log)),
        "R2_orig": float(r2_score(y_true_orig, y_pred_orig)),
    }

scoring = {
    "MAE":  "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAPE": make_scorer(mape_safe, greater_is_better=False),
    "R2":   "r2",
}

def make_preprocessor(num_cols, ord_cols, cat_cols, treat_ord_as_num=True):
    transformers = []
    if num_cols:
        transformers.append(("num", StandardScaler(), num_cols))
    if ord_cols:
        step = StandardScaler() if treat_ord_as_num else OneHotEncoder(handle_unknown="ignore", drop="first")
        transformers.append(("ord", step, ord_cols))
    if cat_cols:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols))
    return ColumnTransformer(transformers=transformers, remainder="drop")

def to_dense(X):
    return X.toarray() if sp.issparse(X) else np.asarray(X)

def get_feat_names(pipe):
    return pipe.named_steps["preprocess"].get_feature_names_out()

def sig_stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    if p < 0.10:  return "."
    return "ns"

def nested_f_test(ols_restricted, ols_full):
    rss_r, rss_f = ols_restricted.ssr, ols_full.ssr
    df_r,  df_f  = ols_restricted.df_resid, ols_full.df_resid
    q = int(df_r - df_f)
    if q <= 0:
        return np.nan, np.nan, q
    f_stat = ((rss_r - rss_f) / q) / (rss_f / df_f)
    p_val  = float(f_dist.sf(f_stat, q, df_f))
    return float(f_stat), p_val, q

def get_cv_splits():
    splits = []
    for seed in range(N_REPEATS):
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        splits.extend(list(skf.split(X_train, strat_labels)))
    return splits

def short_label(raw):
    for pfx in ("num__","ord__","cat__"):
        if raw.startswith(pfx):
            raw = raw[len(pfx):]
            break
    for col in CAT_COLS:
        if raw.startswith(col + "_"):
            return raw[len(col)+1:]
    return raw

# Model specs
MODEL_SPECS = [
    {"name": "M1", "label": "numeric only",
     "numeric": NUM_COLS, "ordinal": [],       "categorical": []},
    {"name": "M2", "label": "numeric + Size",
     "numeric": NUM_COLS, "ordinal": ["size_availability_award"], "categorical": []},
    {"name": "M3", "label": "numeric + Discount",
     "numeric": NUM_COLS, "ordinal": ["discount_group"],          "categorical": []},
    {"name": "M4", "label": "numeric + URUN ALT GRUBU",
     "numeric": NUM_COLS, "ordinal": [],       "categorical": ["URUN ALT GRUBU"]},
    {"name": "M5", "label": "numeric + both ordinals",
     "numeric": NUM_COLS, "ordinal": ORD_COLS, "categorical": []},
    {"name": "M6", "label": "numeric + ordinal + URUN ALT GRUBU",
     "numeric": NUM_COLS, "ordinal": ORD_COLS, "categorical": ["URUN ALT GRUBU"]},
    {"name": "M7", "label": "numeric + ordinal + LifeStyleGroup",
     "numeric": NUM_COLS, "ordinal": ORD_COLS, "categorical": ["LifeStyleGroup"]},
    {"name": "M8", "label": "numeric + ordinal + ColorGroup",
     "numeric": NUM_COLS, "ordinal": ORD_COLS, "categorical": ["ColorGroup"]},
    {"name": "M9", "label": "full (all features)",
     "numeric": NUM_COLS, "ordinal": ORD_COLS, "categorical": CAT_COLS},
]

NESTED_PAIRS = [
    ("M1", "M2", "size_availability_award"),
    ("M1", "M3", "discount_group"),
    ("M1", "M4", "URUN ALT GRUBU"),
    ("M1", "M5", "both ordinals"),
    ("M2", "M5", "discount_group (given size already in)"),
    ("M3", "M5", "size_availability_award (given discount already in)"),
    ("M4", "M6", "both ordinals (given URUN ALT GRUBU already in)"),
    ("M5", "M6", "URUN ALT GRUBU"),
    ("M5", "M7", "LifeStyleGroup"),
    ("M5", "M8", "ColorGroup"),
    ("M6", "M9", "LifeStyleGroup + ColorGroup (on top of M6)"),
    ("M7", "M9", "URUN ALT GRUBU + ColorGroup (on top of M7)"),
    ("M8", "M9", "URUN ALT GRUBU + LifeStyleGroup (on top of M8)"),
]

# Training
results        = {}
ols_models     = {}
pipelines      = {}
feat_names_map = {}

for spec in MODEL_SPECS:
    mname    = spec["name"]
    num_cols = spec["numeric"]
    ord_cols = spec["ordinal"]
    cat_cols = spec["categorical"]

    preprocessor = make_preprocessor(num_cols, ord_cols, cat_cols, TREAT_ORD_AS_NUM)
    pipe = Pipeline([("preprocess", preprocessor),
                     ("model",      LinearRegression(fit_intercept=True))])

    cv_out = cross_validate(pipe, X_train, y_train,
                            cv=get_cv_splits(), scoring=scoring,
                            n_jobs=-1, return_train_score=True)
    cv_mae      = float(-cv_out["test_MAE"].mean())
    cv_rmse     = float(-cv_out["test_RMSE"].mean())
    cv_mape     = float(-cv_out["test_MAPE"].mean())
    cv_r2       = float( cv_out["test_R2"].mean())
    cv_r2_train = float( cv_out["train_R2"].mean())

    pipe.fit(X_train, y_train)
    y_pred_test_log = pipe.predict(X_test)
    test_m = evaluate(y_test, y_pred_test_log)

    X_tr_prep  = to_dense(pipe.named_steps["preprocess"].transform(X_train))
    feat_names = get_feat_names(pipe)
    X_sm_train = sm.add_constant(X_tr_prep, has_constant="add")
    ols        = sm.OLS(y_train.values, X_sm_train).fit(method="pinv")

    n, k   = len(y_train), X_tr_prep.shape[1]
    adj_r2 = float(1 - (1 - ols.rsquared) * (n - 1) / (n - k - 1))

    results[mname] = {
        "label": spec["label"], "k": k,
        "CV_MAE": cv_mae, "CV_RMSE": cv_rmse, "CV_MAPE": cv_mape,
        "CV_R2": cv_r2, "CV_R2_train": cv_r2_train,
        "TEST_MAE": test_m["MAE"], "TEST_RMSE": test_m["RMSE"],
        "TEST_MAPE": test_m["MAPE"], "TEST_R2": test_m["R2"],
        "TEST_R2_orig": test_m["R2_orig"], "TRAIN_R2": float(ols.rsquared),
        "ADJ_R2": adj_r2, "AIC": float(ols.aic), "BIC": float(ols.bic),
        "F_stat": float(ols.fvalue), "F_pval": float(ols.f_pvalue),
        "overfit_gap": cv_r2_train - cv_r2,
    }
    ols_models[mname]     = ols
    pipelines[mname]      = pipe
    feat_names_map[mname] = feat_names

print(f"Setup complete — Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Models trained: {list(results.keys())}")


Setup complete — Train: (480, 11)  |  Test: (121, 11)
Models trained: ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']


## M9 baseline check

This section assumes your earlier training cells have already created the Full M9 objects such as
`pipelines["M9"]`, `pipe9`, `results`, and related metadata.


In [2]:
# M9 is already trained in the main loop above.

m = "M9"
r9 = results[m]
print("=" * 80)
print(f"  BASELINE: {m} — {r9['label']}")
print("=" * 80)
print(f"  k (features after encoding) : {r9['k']}")
print(f"  TEST RMSE                   : {r9['TEST_RMSE']:.4f}")
print(f"  TEST R²                     : {r9['TEST_R2']:.4f}")
print(f"  Adj. R²                     : {r9['ADJ_R2']:.4f}")
print(f"  AIC                         : {r9['AIC']:.2f}")
print(f"  Overfit gap (CV_train-CV_val): {r9['overfit_gap']:.4f}")
print()
print("  Feature groups in M9:")
spec9 = next(s for s in MODEL_SPECS if s["name"] == "M9")
for group, cols in [("Numeric",     spec9["numeric"]),
                    ("Ordinal",      spec9["ordinal"]),
                    ("Categorical",  spec9["categorical"])]:
    if cols:
        print(f"    {group}: {cols}")
print()

  BASELINE: M9 — full (all features)
  k (features after encoding) : 47
  TEST RMSE                   : 37.8433
  TEST R²                     : 0.4685
  Adj. R²                     : 0.4908
  AIC                         : 942.53
  Overfit gap (CV_train-CV_val): 0.2331

  Feature groups in M9:
    Numeric: ['ETIKET']
    Ordinal: ['size_availability_award', 'discount_group']
    Categorical: ['URUN ALT GRUBU', 'LifeStyleGroup', 'ColorGroup']



## Final scoring and export

In [21]:
# FINAL SCORING ENGINE — FULL M9 ONLY

# 1) Resolve the Full M9 pipeline and rename as SSCORER
if "pipe9" in globals():
    SSCORER = pipe9
elif "pipelines" in globals() and "M9" in pipelines:
    SSCORER = pipelines["M9"]
else:
    raise NameError("Full M9 pipeline was not found. Run the training cells that create pipe9 or pipelines['M9'] first.")  #KESİN VAR, SİLİNEBİLİR

print("Using scoring engine:", SSCORER)

# 2) New data
scored_df = pd.read_excel("../../data/01_raw/product_arrival.xlsx").copy()
scored_df = scored_df.dropna(how="all").copy()

# remove rows that are empty across main model-driving fields
key_cols = [
    c for c in [
        "ETIKET",
        "URUN ALT GRUBU",
        "LifeStyleGroup",
        "ColorGroup",
        "size_availability_award",
        "discount_group"
    ]
    if c in scored_df.columns]

if key_cols:
    scored_df = scored_df.dropna(subset=key_cols, how="all").copy()

print("Shape after cleaning empty rows:", scored_df.shape)

# keep original id if exists
if "ItemOption" in scored_df.columns:
    scored_df["original_ItemOption"] = scored_df["ItemOption"]
elif "ItemOption" in scored_df.columns:
    scored_df["original_ItemOption"] = scored_df["ItemOption"]

# 3) MATCH TRAINING PREPROCESSING
# ETIKET was log1p-transformed before model training
if "ETIKET" in scored_df.columns:
    scored_df["ETIKET"] = pd.to_numeric(scored_df["ETIKET"], errors="coerce")
    scored_df["ETIKET"] = scored_df["ETIKET"].clip(lower=0)
    scored_df["ETIKET"] = np.log1p(scored_df["ETIKET"])

# 4) Missing values YOK AMA OLSUN
num_cols = scored_df.select_dtypes(include=[np.number]).columns
cat_cols = scored_df.select_dtypes(exclude=[np.number]).columns

scored_df[num_cols] = scored_df[num_cols].fillna(0)
scored_df[cat_cols] = scored_df[cat_cols].fillna("Unknown")

# strip spaces in object columns to avoid fake categories like "Black " vs "Black" YOK AMA OLSUN
for col in cat_cols:
    scored_df[col] = scored_df[col].astype(str).str.strip()


# 5) Predict
scored_df["pred_log_full_M9"] = SSCORER.predict(scored_df)
scored_df["pred_orig_full_M9"] = np.expm1(scored_df["pred_log_full_M9"])
scored_df["pred_orig_full_M9"] = scored_df["pred_orig_full_M9"].clip(lower=0)

# 6) ORIGINAL ATTITUDE: proportional 0-1 scoring, safe for negatives
def normalize_max_ratio_safe(series, eps=1e-9):
    s = pd.Series(series).astype(float).copy()

    min_val = s.min()
    if min_val <= 0:
        s = s - min_val + eps

    max_val = s.max()
    if max_val <= 0:
        return pd.Series([0.0] * len(s), index=s.index)

    return s / max_val

scored_df["saleability_score_full_M9_log"] = normalize_max_ratio_safe(
    scored_df["pred_log_full_M9"]
)

scored_df["saleability_score_full_M9_orig"] = normalize_max_ratio_safe(
    scored_df["pred_orig_full_M9"]
)

# 7) Required nd0, nd1, nd2...
item_codes = [f"nd{i}" for i in range(len(scored_df))]

if "ItemOption" in scored_df.columns:
    scored_df["ItemOption"] = item_codes
elif "ItemOption" in scored_df.columns:
    scored_df["ItemOption"] = item_codes
else:
    scored_df.insert(0, "ItemOption", item_codes)

# 8) Final export table
id_col = "ItemOption" if "ItemOption" in scored_df.columns else "ItemOption"

score_cols = [ "pred_log_full_M9", "saleability_score_full_M9_log", "pred_orig_full_M9", "saleability_score_full_M9_orig"]
feature_cols = [col for col in scored_df.columns if col not in score_cols]
saleability_table = scored_df[feature_cols + score_cols].copy()

saleability_table = saleability_table.sort_values(
    by="saleability_score_full_M9_orig",
    ascending=False).reset_index(drop=True)

saleability_table.head()

Using scoring engine: Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['ETIKET']),
                                                 ('ord',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['size_availability_award',
                                                   'discount_group']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['URUN ALT GRUBU',
                                                   'LifeStyleGroup',
                                                   'ColorGroup'])])),
  

,ItemOption,URUN ALT GRUBU,LifeStyleGroup,ETIKET,ColorGroup,active_weeks,size_unique_count,size_availability_award,discount_group,discount,pred_log_full_M9,saleability_score_full_M9_log,pred_orig_full_M9,saleability_score_full_M9_orig
0,nd434,TISORT,Mono,6.551080,brown,5.0,4.0,1.0,1.0,0.18,5.143132,1.000000,170.251337,1.000000
1,nd448,TISORT,Business,6.907755,beige,3.0,3.0,1.0,1.0,0.14,4.910135,0.954697,134.657691,0.790935
2,nd468,TISORT,Mono,6.551080,burgundy,5.0,3.0,1.0,2.0,0.34,4.831534,0.939415,124.403192,0.730703
3,nd266,PANTOLON,Essential,7.495542,burgundy,9.0,5.0,2.0,1.0,0.08,4.708262,0.915446,109.859328,0.645277
4,nd85,TRIKO,Essential,7.438384,black,9.0,5.0,2.0,1.0,0.15,4.698276,0.913505,108.757743,0.638807


In [22]:
saleability_table.to_excel("../../data/02_interim/saleability_scores_full_M9.xlsx", index=False)
print("Exported: saleability_scores_full_M9.xlsx")

Exported: saleability_scores_full_M9.xlsx
